# Telegram Save Restricted Content Bot - Kaggle Runner

**Muhim:** Kaggle Settings da **Internet** yoqilgan bo'lishi kerak.

**Qadamlar:**
1. Loyihani yuklash (GitHub yoki Dataset)
2. Kutubxonalarni o'rnatish
3. Sozlamalar
4. Botni ishga tushirish

**To'xtatish:** Katak yonidagi stop tugmasini bosing.

---
## 1. Loyihani yuklash

In [ ]:
# ============================================================
# LOYIHA MANBASI - birini tanlang
# ============================================================
PROJECT_SOURCE = "github"  # "github" yoki "upload"

# GitHub sozlamalari
GITHUB_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # <-- O'ZGARTIRING
BRANCH = "main"
GITHUB_TOKEN = ""  # Shaxsiy repo: ghp_xxxx

# ============================================================
import os, sys, shutil, glob

PROJECT_DIR = "/kaggle/working/project"

def find_project_root(base):
    """config.py + core/ papkasi bor joyni topadi - necha qatlam ichida bo'lmasin."""
    if os.path.isfile(os.path.join(base, "config.py")) and os.path.isdir(os.path.join(base, "core")):
        return base
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ("venv", "__pycache__", ".git", "node_modules")]
        if "config.py" in files and "core" in dirs:
            return root
    return None

if PROJECT_SOURCE == "github":
    if os.path.isfile(os.path.join(PROJECT_DIR, "config.py")):
        print(f"Loyiha allaqachon mavjud: {PROJECT_DIR}")
    else:
        if os.path.exists(PROJECT_DIR):
            shutil.rmtree(PROJECT_DIR)
        url = GITHUB_URL
        if GITHUB_TOKEN:
            url = GITHUB_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
        ret = os.system(f"git clone --branch {BRANCH} --depth 1 {url} {PROJECT_DIR}")
        if ret != 0:
            raise RuntimeError("Git clone xatosi!")
        real = find_project_root(PROJECT_DIR)
        if real and real != PROJECT_DIR:
            tmp = PROJECT_DIR + "_tmp"
            shutil.move(real, tmp)
            shutil.rmtree(PROJECT_DIR)
            shutil.move(tmp, PROJECT_DIR)
        print("GitHub'dan klonlandi")

elif PROJECT_SOURCE == "upload":
    # Kaggle: Add Data > Upload orqali ZIP yuklang
    kaggle_input = "/kaggle/input"
    datasets = []
    if os.path.isdir(kaggle_input):
        datasets = [d for d in os.listdir(kaggle_input)
                    if os.path.isdir(os.path.join(kaggle_input, d))]
    if not datasets:
        raise RuntimeError(
            "Dataset topilmadi! 'Add Data' > 'Upload' orqali loyiha arxivini yuklang."
        )
    ds_path = os.path.join(kaggle_input, datasets[0])
    print(f"Dataset: {datasets[0]}")

    # Arxiv bormi?
    archives = glob.glob(os.path.join(ds_path, "*.zip")) + \
               glob.glob(os.path.join(ds_path, "*.tar.gz")) + \
               glob.glob(os.path.join(ds_path, "*.tgz"))

    if archives:
        archive = archives[0]
        extract_tmp = "/kaggle/working/_extract_tmp"
        if os.path.exists(extract_tmp):
            shutil.rmtree(extract_tmp)
        os.makedirs(extract_tmp)
        if archive.endswith(".zip"):
            import zipfile
            with zipfile.ZipFile(archive, "r") as zf:
                zf.extractall(extract_tmp)
        else:
            import tarfile
            with tarfile.open(archive, "r:*") as tf:
                tf.extractall(extract_tmp)
        real = find_project_root(extract_tmp)
        if real is None:
            raise RuntimeError("Arxiv ichidan config.py + core/ topilmadi!")
        if os.path.exists(PROJECT_DIR):
            shutil.rmtree(PROJECT_DIR)
        shutil.copytree(real, PROJECT_DIR)
        shutil.rmtree(extract_tmp)
    else:
        # Oddiy papka sifatida
        real = find_project_root(ds_path)
        if real is None:
            raise RuntimeError("Dataset ichidan config.py + core/ topilmadi!")
        if os.path.exists(PROJECT_DIR):
            shutil.rmtree(PROJECT_DIR)
        shutil.copytree(real, PROJECT_DIR)
    print("Dataset dan ochildi")

else:
    raise ValueError(f"Noto'g'ri PROJECT_SOURCE: {PROJECT_SOURCE}")

assert os.path.isfile(os.path.join(PROJECT_DIR, "config.py")), "config.py topilmadi!"
assert os.path.isdir(os.path.join(PROJECT_DIR, "core")), "core/ topilmadi!"
print(f"Loyiha tayyor: {PROJECT_DIR}")

---
## 2. Kutubxonalarni o'rnatish

In [ ]:
import subprocess, sys, os

PROJECT_DIR = "/kaggle/working/project"
req_file = os.path.join(PROJECT_DIR, "requirements.txt")
assert os.path.exists(req_file), "requirements.txt topilmadi!"

print("Kutubxonalar o'rnatilmoqda...")
ret = subprocess.call([
    sys.executable, "-m", "pip", "install",
    "--no-cache-dir", "--prefer-binary", "--progress-bar", "off",
    "-r", req_file
])
if ret != 0:
    print("OGOHLANTIRISH: Ba'zi kutubxonalar o'rnatilmagan bo'lishi mumkin.")

try:
    import uvloop
    print(f"uvloop: {uvloop.__version__}")
except ImportError:
    print("uvloop: o'rnatilmagan")

try:
    import nest_asyncio
    print("nest_asyncio: OK")
except ImportError:
    subprocess.call([sys.executable, "-m", "pip", "install", "-q", "nest_asyncio"])

print(f"\nPython: {sys.version}")
print("O'rnatish tugadi.")

---
## 3. Sozlamalar

**Xavfsiz usul:** `Add-ons > Secrets` da:
- `BOT_TOKEN`, `API_ID`, `API_HASH`, `OWNER_ID`, `DB_URI` qo'shing

**Oddiy usul:** Quyidagi maydonlarga qo'lda yozing.

In [ ]:
import os

PROJECT_DIR = "/kaggle/working/project"
_keys = ["BOT_TOKEN", "API_ID", "API_HASH", "OWNER_ID", "DB_URI", "OWNER_USERNAME"]
_loaded = set()

# ============================================================
# 1-MANBA: .env fayl (ZIP bilan yuklangan loyiha ichida)
# ============================================================
_env_path = os.path.join(PROJECT_DIR, ".env")
if os.path.isfile(_env_path):
    print(f".env fayl topildi: {_env_path}")
    with open(_env_path) as _f:
        for _line in _f:
            _line = _line.strip()
            if not _line or _line.startswith("#") or "=" not in _line:
                continue
            _k, _v = _line.split("=", 1)
            _k, _v = _k.strip(), _v.strip()
            if _k in _keys and _v:
                os.environ[_k] = _v
                _loaded.add(_k)
    if _loaded:
        print(f".env dan yuklandi: {', '.join(sorted(_loaded))}")

# ============================================================
# 2-MANBA: Kaggle Secrets (qolgan kalitlar uchun)
# ============================================================
try:
    from kaggle_secrets import UserSecretsClient
    _sc = UserSecretsClient()
    for _k in _keys:
        if _k in _loaded:
            continue
        try:
            _v = _sc.get_secret(_k)
            if _v:
                os.environ[_k] = str(_v)
                _loaded.add(_k)
        except Exception:
            pass
    if _loaded:
        print(f"Kaggle Secrets dan yuklandi: {', '.join(sorted(_loaded))}")
except Exception:
    pass

# ============================================================
# 3-MANBA: Qo'lda kiritish (hali bo'sh qolganlar uchun)
# ============================================================
if "BOT_TOKEN" not in _loaded and not os.environ.get("BOT_TOKEN"):
    os.environ["BOT_TOKEN"]   = ""  # <-- BotFather dan
if "API_ID" not in _loaded and not os.environ.get("API_ID"):
    os.environ["API_ID"]      = ""  # <-- my.telegram.org dan
if "API_HASH" not in _loaded and not os.environ.get("API_HASH"):
    os.environ["API_HASH"]    = ""  # <-- my.telegram.org dan
if "OWNER_ID" not in _loaded and not os.environ.get("OWNER_ID"):
    os.environ["OWNER_ID"]    = ""  # <-- Telegram ID
if "DB_URI" not in _loaded and not os.environ.get("DB_URI"):
    os.environ["DB_URI"]      = ""  # <-- MongoDB URI

# ============================================================
# Tekshirish
# ============================================================
_ok = True
for _k in ["BOT_TOKEN", "API_ID", "API_HASH"]:
    _v = os.environ.get(_k, "")
    if not _v or _v == "0":
        print(f"XATO: {_k} bo'sh! .env ga yozing, Secrets ga qo'shing, yoki yuqorida to'ldiring.")
        _ok = False
    else:
        print(f"{_k}: ...{_v[-6:]}")

for _k in ["OWNER_ID", "DB_URI"]:
    _v = os.environ.get(_k, "")
    if _v and _v != "0":
        print(f"{_k}: ...{_v[-6:]}")
    else:
        print(f"{_k}: (ixtiyoriy)")

if not _ok:
    raise SystemExit("Majburiy sozlamalar to'ldirilmagan!")
print("\nSozlamalar tayyor.")

---
## 4. Botni ishga tushirish

To'xtatish: katak yonidagi **stop** tugmasi.

In [ ]:
import os, sys, asyncio, logging, time

PROJECT_DIR = "/kaggle/working/project"
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

for d in ["sessions", "downloads/temp", "logs", "data"]:
    os.makedirs(d, exist_ok=True)

# Asyncio patch
import nest_asyncio
nest_asyncio.apply()

from core.compat import configure_event_loop_policy
configure_event_loop_policy()

# Logging
logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
for _n in ["pymongo", "pyrogram", "pyrogram.session",
           "pyrogram.connection", "pyrogram.dispatcher"]:
    logging.getLogger(_n).setLevel(logging.ERROR)
logging.getLogger("TechVJ").setLevel(logging.INFO)
logging.getLogger("core").setLevel(logging.INFO)

# Kaggle runtime tuning (konservativ - bandwidth limit)
try:
    from core.downloader import worker as _w
    _w.DEFAULT_WORKER_COUNT = 8
    _w.MAX_WORKER_COUNT = 32
    _w.MIN_WORKER_COUNT = 4
except Exception:
    pass

try:
    from core.downloader import adaptive_engine as _ae
    _ae.NETWORK_CHUNK_SIZE = 256 * 1024
    _ae.CHECKPOINT_BYTES_MEDIUM = 10 * 1024 * 1024
    _ae.CHECKPOINT_BYTES_LARGE = 50 * 1024 * 1024
except Exception:
    pass

# Bot
from main import Bot
from pyrogram import idle

bot = Bot()

async def run_bot():
    t0 = time.time()
    try:
        await bot.start()
        print("=" * 50)
        print(f"Bot ishga tushdi! ({time.time()-t0:.1f}s)")
        print("To'xtatish: stop tugmasi")
        print("=" * 50)
        await idle()
    except (KeyboardInterrupt, asyncio.CancelledError):
        print("\nBot to'xtatilmoqda...")
    finally:
        try:
            await bot.stop()
        except Exception:
            pass
        print("Bot to'xtadi.")

loop = asyncio.get_event_loop()
try:
    loop.run_until_complete(run_bot())
except KeyboardInterrupt:
    print("\nFoydalanuvchi to'xtatdi.")
except Exception as e:
    print(f"Xato: {e}")